# 🥈 Template: Silver Layer - Transformações com SCD Tipo 2

## 📋 Objetivo
Este template implementa transformações da camada Silver com Slowly Changing Dimensions (SCD) Tipo 2 para rastreamento histórico.

## 🎯 Características do SCD Tipo 2
- **Versionamento**: Mantém histórico completo de mudanças
- **Rastreabilidade**: Sabe quando e como os dados mudaram
- **Integridade**: Preserva relacionamentos temporais
- **Performance**: Otimizado para consultas analíticas

## 🔧 Configurações
Adapte as variáveis abaixo para seu projeto específico.

In [ ]:
# 📦 Importações e Configurações Iniciais
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from delta.tables import DeltaTable
import logging
from datetime import datetime, date
from functools import reduce

# Configurar logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# Inicializar Spark session
spark = SparkSession.builder.appName("SilverTransformations").getOrCreate()
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "true")
spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "true")

print("✅ Configurações iniciais completas")

In [ ]:
# 🎯 Configurações do Projeto
# TODO: Personalize estas configurações para seu projeto

# Configurações do Lakehouse
LAKEHOUSE_PATH = "/lakehouse/default/"
BRONZE_PATH = f"{LAKEHOUSE_PATH}Tables/bronze"
SILVER_PATH = f"{LAKEHOUSE_PATH}Tables/silver"

# Configurações SCD Tipo 2
DEFAULT_START_DATE = date(1900, 1, 1)
DEFAULT_END_DATE = date(9999, 12, 31)
PROCESSING_DATE = date.today()

# Tabelas para processar com configurações SCD
# TODO: Definir tabelas específicas do seu domínio
TABLES_CONFIG = {
    "customers": {
        "natural_key": "customer_id",  # Chave natural (business key)
        "scd_columns": ["customer_name", "email", "phone", "address"],  # Colunas que devem rastrear mudanças
        "partition_columns": ["effective_year"],
        "business_rules": True,  # Aplicar regras de negócio específicas
        "data_quality_checks": True
    },
    "orders": {
        "natural_key": "order_id",
        "scd_columns": ["order_status", "payment_method", "shipping_address"],
        "partition_columns": ["effective_year"],
        "business_rules": True,
        "data_quality_checks": True
    },
    "products": {
        "natural_key": "product_id",
        "scd_columns": ["product_name", "category", "price", "description"],
        "partition_columns": ["effective_year"],
        "business_rules": True,
        "data_quality_checks": True
    }
}

print(f"📅 Data de processamento: {PROCESSING_DATE}")
print(f"🥉 Caminho Bronze: {BRONZE_PATH}")
print(f"🥈 Caminho Silver: {SILVER_PATH}")

In [ ]:
# 🔧 Funções SCD Tipo 2

def add_scd2_columns(df):
    """
    Adiciona colunas padrão do SCD Tipo 2
    
    Args:
        df: DataFrame PySpark
    
    Returns:
        DataFrame com colunas SCD Tipo 2
    """
    return df.withColumn("effective_date", lit(PROCESSING_DATE).cast("date")) \
             .withColumn("end_date", lit(DEFAULT_END_DATE).cast("date")) \
             .withColumn("is_current", lit(True)) \
             .withColumn("version", lit(1)) \
             .withColumn("created_date", current_timestamp()) \
             .withColumn("updated_date", current_timestamp()) \
             .withColumn("effective_year", year(col("effective_date"))) \
             .withColumn("effective_month", month(col("effective_date")))

def generate_surrogate_key(df, natural_key_col, prefix="SK"):
    """
    Gera chave surrogate otimizada para SCD Tipo 2
    
    Args:
        df: DataFrame PySpark
        natural_key_col: Nome da coluna chave natural
        prefix: Prefixo para a chave surrogate
    
    Returns:
        DataFrame com chave surrogate
    """
    # Usar hash da chave natural + timestamp para garantir unicidade
    return df.withColumn(
        f"{prefix}_{natural_key_col}",
        concat(
            lit(prefix + "_"),
            abs(hash(concat(col(natural_key_col), col("effective_date"), col("version")))).cast("string")
        )
    )

def apply_data_quality_checks(df, table_name, config):
    """
    Aplica validações de qualidade dos dados
    
    Args:
        df: DataFrame PySpark
        table_name: Nome da tabela
        config: Configuração da tabela
    
    Returns:
        DataFrame limpo
    """
    initial_count = df.count()
    
    # 1. Remover registros com chave natural nula
    natural_key = config['natural_key']
    df_clean = df.filter(col(natural_key).isNotNull() & (col(natural_key) != ""))
    
    # 2. Remover duplicatas baseadas na chave natural e data efetiva
    df_clean = df_clean.dropDuplicates([natural_key, "effective_date"])
    
    # 3. TODO: Adicionar validações específicas do domínio
    # Exemplo para e-commerce:
    if table_name == "customers":
        # Validar formato de email
        df_clean = df_clean.filter(
            col("email").isNull() | 
            col("email").rlike(r"^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}$")
        )
    
    elif table_name == "orders":
        # Validar datas de pedido
        df_clean = df_clean.filter(
            col("order_date").isNull() | 
            (col("order_date") <= current_date())
        )
    
    elif table_name == "products":
        # Validar preços positivos
        df_clean = df_clean.filter(
            col("price").isNull() | 
            (col("price") >= 0)
        )
    
    final_count = df_clean.count()
    rejected_count = initial_count - final_count
    
    logger.info(f"📊 {table_name} - Qualidade: {initial_count} → {final_count} ({rejected_count} rejeitados)")
    
    return df_clean

def apply_business_rules(df, table_name):
    """
    Aplica regras de negócio específicas do domínio
    
    Args:
        df: DataFrame PySpark
        table_name: Nome da tabela
    
    Returns:
        DataFrame com regras aplicadas
    """
    # TODO: Implementar regras específicas do seu domínio
    
    if table_name == "customers":
        # Exemplo: Padronizar dados de cliente
        df = df.withColumn("customer_name", trim(upper(col("customer_name")))) \
               .withColumn("email", lower(trim(col("email")))) \
               .withColumn("phone", regexp_replace(col("phone"), r"[^0-9]", "")) \
               .withColumn("customer_segment", 
                          when(col("total_orders") >= 10, "VIP")
                          .when(col("total_orders") >= 5, "Regular")
                          .otherwise("New"))
    
    elif table_name == "orders":
        # Exemplo: Calcular métricas de pedido
        df = df.withColumn("order_value_category",
                          when(col("order_value") >= 1000, "High")
                          .when(col("order_value") >= 100, "Medium")
                          .otherwise("Low")) \
               .withColumn("is_weekend_order",
                          dayofweek(col("order_date")).isin([1, 7])) \
               .withColumn("days_to_delivery",
                          datediff(col("delivery_date"), col("order_date")))
    
    elif table_name == "products":
        # Exemplo: Categorizar produtos
        df = df.withColumn("price_category",
                          when(col("price") >= 500, "Premium")
                          .when(col("price") >= 100, "Standard")
                          .otherwise("Economy")) \
               .withColumn("product_name_clean", trim(upper(col("product_name")))) \
               .withColumn("has_description", col("description").isNotNull())
    
    return df

def perform_scd2_merge(df_new, table_name, config):
    """
    Executa merge SCD Tipo 2 com dados existentes
    
    Args:
        df_new: DataFrame com novos dados
        table_name: Nome da tabela
        config: Configuração da tabela
    
    Returns:
        DataFrame resultado do merge
    """
    silver_table_path = f"{SILVER_PATH}/{table_name}"
    natural_key = config['natural_key']
    scd_columns = config['scd_columns']
    
    # Verificar se a tabela já existe
    try:
        if DeltaTable.isDeltaTable(spark, silver_table_path):
            # Tabela existe - fazer merge SCD Tipo 2
            existing_table = DeltaTable.forPath(spark, silver_table_path)
            
            # Identificar registros que mudaram
            df_existing = existing_table.toDF().filter(col("is_current") == True)
            
            # Criar condição de mudança baseada nas colunas SCD
            change_conditions = []
            for scd_col in scd_columns:
                change_conditions.append(
                    f"existing.{scd_col} != new.{scd_col} OR "
                    f"(existing.{scd_col} IS NULL AND new.{scd_col} IS NOT NULL) OR "
                    f"(existing.{scd_col} IS NOT NULL AND new.{scd_col} IS NULL)"
                )
            
            change_condition = " OR ".join([f"({cond})" for cond in change_conditions])
            
            # 1. Atualizar registros existentes que mudaram (marcar como não-correntes)
            existing_table.alias("existing").merge(
                df_new.alias("new"),
                f"existing.{natural_key} = new.{natural_key} AND existing.is_current = true"
            ).whenMatchedUpdate(
                condition=change_condition,
                set={
                    "is_current": "false",
                    "end_date": f"date_sub(new.effective_date, 1)",
                    "updated_date": "current_timestamp()"
                }
            ).execute()
            
            # 2. Inserir novos registros
            # Pegar próxima versão para registros que existem
            max_versions = df_existing.groupBy(natural_key).agg(
                max("version").alias("max_version")
            )
            
            df_with_versions = df_new.join(
                max_versions,
                natural_key,
                "left"
            ).withColumn(
                "version",
                when(col("max_version").isNull(), 1)
                .otherwise(col("max_version") + 1)
            ).drop("max_version")
            
            # Gerar nova chave surrogate
            df_with_versions = generate_surrogate_key(df_with_versions, natural_key)
            
            # Inserir apenas registros novos ou que mudaram
            existing_table.alias("existing").merge(
                df_with_versions.alias("new"),
                f"existing.{natural_key} = new.{natural_key} AND existing.is_current = true"
            ).whenNotMatchedInsertAll().execute()
            
            logger.info(f"✅ {table_name} - Merge SCD Tipo 2 concluído")
            
        else:
            # Primeira carga - criar tabela
            df_with_sk = generate_surrogate_key(df_new, natural_key)
            
            df_with_sk.write \
                .mode("overwrite") \
                .partitionBy(*config.get('partition_columns', [])) \
                .option("mergeSchema", "true") \
                .format("delta") \
                .save(silver_table_path)
            
            logger.info(f"✅ {table_name} - Primeira carga concluída")
    
    except Exception as e:
        logger.error(f"❌ Erro no merge SCD2 para {table_name}: {str(e)}")
        raise
    
    # Registrar tabela no metastore
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS silver_{table_name}
        USING DELTA
        LOCATION '{silver_table_path}'
    """)
    
    return spark.read.format("delta").load(silver_table_path)

print("🔧 Funções SCD Tipo 2 definidas")

In [ ]:
# 🥈 Processamento das Tabelas Silver

silver_summary = []

for table_name, config in TABLES_CONFIG.items():
    try:
        logger.info(f"🔄 Processando tabela Silver: {table_name}")
        
        # 1. Carregar dados da camada Bronze
        bronze_table_name = f"bronze_{table_name}"
        df_bronze = spark.table(bronze_table_name)
        
        initial_count = df_bronze.count()
        logger.info(f"📊 {table_name} - Registros Bronze: {initial_count:,}")
        
        # 2. Aplicar validações de qualidade
        if config.get('data_quality_checks', False):
            df_clean = apply_data_quality_checks(df_bronze, table_name, config)
        else:
            df_clean = df_bronze
        
        # 3. Aplicar regras de negócio
        if config.get('business_rules', False):
            df_transformed = apply_business_rules(df_clean, table_name)
        else:
            df_transformed = df_clean
        
        # 4. Adicionar colunas SCD Tipo 2
        df_scd2 = add_scd2_columns(df_transformed)
        
        # 5. Executar merge SCD Tipo 2
        df_silver = perform_scd2_merge(df_scd2, table_name, config)
        
        final_count = df_silver.filter(col("is_current") == True).count()
        total_historical = df_silver.count()
        
        # 6. Coletar métricas
        summary = {
            "table_name": table_name,
            "bronze_records": initial_count,
            "current_records": final_count,
            "total_historical": total_historical,
            "historical_versions": total_historical - final_count,
            "processing_date": PROCESSING_DATE
        }
        silver_summary.append(summary)
        
        logger.info(f"✅ {table_name} - Processamento concluído: {final_count:,} correntes, {total_historical:,} total")
        
        # 7. Mostrar amostra dos dados Silver
        print(f"\n📋 Amostra da tabela silver_{table_name} (registros correntes):")
        df_silver.filter(col("is_current") == True).limit(5).display()
        
    except Exception as e:
        logger.error(f"❌ Erro ao processar {table_name}: {str(e)}")
        # TODO: Implementar estratégia de recuperação de erro
        continue

print("\n🎉 Processamento da camada Silver concluído!")

In [ ]:
# 📊 Relatório de Transformações Silver

print("\n📊 RELATÓRIO DE TRANSFORMAÇÕES - SILVER LAYER")
print("=" * 60)

# Converter para DataFrame para melhor visualização
summary_df = spark.createDataFrame(
    [Row(**summary) for summary in silver_summary]
)

# Mostrar resumo das transformações
summary_df.select(
    "table_name",
    "bronze_records",
    "current_records",
    "historical_versions",
    "total_historical",
    "processing_date"
).display()

# Calcular estatísticas gerais
total_bronze = sum([s['bronze_records'] for s in silver_summary])
total_current = sum([s['current_records'] for s in silver_summary])
total_historical = sum([s['total_historical'] for s in silver_summary])

print(f"\n📋 RESUMO GERAL:")
print(f"   📊 Total registros Bronze processados: {total_bronze:,}")
print(f"   🎯 Total registros Silver correntes: {total_current:,}")
print(f"   📚 Total registros históricos: {total_historical:,}")
print(f"   📅 Data de processamento: {PROCESSING_DATE}")

# Mostrar eficiência das transformações
if total_bronze > 0:
    efficiency = (total_current / total_bronze) * 100
    print(f"   ⚡ Eficiência das transformações: {efficiency:.2f}%")

# Identificar tabelas com muitas versões históricas
high_versioning = [s for s in silver_summary if s['historical_versions'] > s['current_records'] * 0.1]
if high_versioning:
    print(f"\n📈 Tabelas com alto versionamento (>10% de registros históricos):")
    for table in high_versioning:
        version_pct = (table['historical_versions'] / table['total_historical']) * 100
        print(f"   - {table['table_name']}: {version_pct:.1f}% histórico")
else:
    print(f"\n✅ Versionamento histórico está equilibrado")

In [ ]:
# 🔍 Validação SCD Tipo 2

print("\n🔍 VALIDAÇÃO SCD TIPO 2")
print("=" * 40)

for table_name, config in TABLES_CONFIG.items():
    try:
        table_full_name = f"silver_{table_name}"
        natural_key = config['natural_key']
        
        print(f"\n📋 Validando {table_name}:")
        
        # 1. Verificar integridade das datas
        invalid_dates = spark.sql(f"""
            SELECT COUNT(*) as count
            FROM {table_full_name}
            WHERE effective_date > end_date
        """).collect()[0].count
        
        if invalid_dates == 0:
            print(f"   ✅ Datas válidas (effective_date <= end_date)")
        else:
            print(f"   ❌ {invalid_dates} registros com datas inválidas")
        
        # 2. Verificar unicidade de registros correntes
        current_duplicates = spark.sql(f"""
            SELECT {natural_key}, COUNT(*) as count
            FROM {table_full_name}
            WHERE is_current = true
            GROUP BY {natural_key}
            HAVING COUNT(*) > 1
        """).count()
        
        if current_duplicates == 0:
            print(f"   ✅ Registros correntes únicos por chave natural")
        else:
            print(f"   ❌ {current_duplicates} chaves naturais com múltiplos registros correntes")
        
        # 3. Verificar sequência de versões
        version_gaps = spark.sql(f"""
            WITH version_check AS (
                SELECT {natural_key}, version,
                       LAG(version) OVER (PARTITION BY {natural_key} ORDER BY version) as prev_version
                FROM {table_full_name}
            )
            SELECT COUNT(*) as count
            FROM version_check
            WHERE prev_version IS NOT NULL AND version != prev_version + 1
        """).collect()[0].count
        
        if version_gaps == 0:
            print(f"   ✅ Sequência de versões íntegra")
        else:
            print(f"   ⚠️  {version_gaps} gaps na sequência de versões")
        
        # 4. Verificar sobreposição de períodos
        overlapping_periods = spark.sql(f"""
            WITH period_check AS (
                SELECT {natural_key}, version, effective_date, end_date,
                       LEAD(effective_date) OVER (PARTITION BY {natural_key} ORDER BY version) as next_effective
                FROM {table_full_name}
            )
            SELECT COUNT(*) as count
            FROM period_check
            WHERE next_effective IS NOT NULL AND end_date >= next_effective
        """).collect()[0].count
        
        if overlapping_periods == 0:
            print(f"   ✅ Sem sobreposição de períodos")
        else:
            print(f"   ❌ {overlapping_periods} períodos sobrepostos")
        
        # 5. Estatísticas de versionamento
        version_stats = spark.sql(f"""
            SELECT 
                MAX(version) as max_version,
                AVG(version) as avg_version,
                COUNT(DISTINCT {natural_key}) as unique_keys
            FROM {table_full_name}
        """).collect()[0]
        
        print(f"   📊 Versão máxima: {version_stats.max_version}")
        print(f"   📊 Versão média: {version_stats.avg_version:.2f}")
        print(f"   📊 Chaves únicas: {version_stats.unique_keys:,}")
        
    except Exception as e:
        print(f"   ❌ Erro na validação: {str(e)}")

print("\n🎉 Validação SCD Tipo 2 concluída!")
print("\n➡️  Próximo passo: Processar camada Gold com dimensões e fatos")

## 📚 Próximos Passos

1. **Personalização**: Adapte as regras de negócio para seu domínio específico
2. **Otimização**: Configure particionamento e Z-ordering adequados
3. **Monitoramento**: Implemente alertas para falhas de qualidade
4. **Gold Layer**: Use o template Gold para criar dimensões e fatos

## 🔧 Customizações Avançadas

### Para Streaming SCD Tipo 2:
```python
# Configurar streaming para updates incrementais
df_stream = spark.readStream \
    .format("cloudFiles") \
    .option("cloudFiles.format", "json") \
    .load("/path/to/streaming/data")

# Aplicar SCD Tipo 2 em modo streaming
query = df_stream.writeStream \
    .foreachBatch(lambda batch, epoch: process_scd2_batch(batch, epoch)) \
    .outputMode("append") \
    .start()
```

### Para Detecção de Mudanças por Hash:
```python
def detect_changes_by_hash(df, natural_key, scd_columns):
    # Criar hash dos valores SCD
    df_with_hash = df.withColumn(
        "scd_hash",
        sha2(concat_ws("|", *[coalesce(col(c), lit("")).cast("string") for c in scd_columns]), 256)
    )
    
    # Comparar com hash anterior
    return df_with_hash.filter(
        col("scd_hash") != col("previous_hash")
    )
```

### Para SCD Tipo 3 (Limited History):
```python
def add_scd3_columns(df, tracked_column):
    return df.withColumn(f"{tracked_column}_current", col(tracked_column)) \
             .withColumn(f"{tracked_column}_previous", lit(None)) \
             .withColumn(f"{tracked_column}_changed_date", current_date())
```